### Purpose
This notebook compares the baseline model against the hybrid model and measures whether hybrid model improves predictive performance.
<br> --- --- --- <br>
Este notebook compara o modelo baseline com o modelo híbrido e mede se o modelo hibrido melhora o desempenho preditivo.
<br>
📝 Language: Technical documentation is maintained in English to ensure consistency and ease of maintenance. Bilingual support (Portuguese/English) is available exclusively within the notebooks.
<br> --- --- --- <br>
📝 Idioma: A documentação é mantida em inglês para garantir consistência técnica e evitar retrabalho na documentação. Isto aplica-se somente aos notebooks.

### Main idea
The notebook builds a structured comparison between:
- a baseline LightGBM model trained only on engineered features
- a hybrid LightGBM model trained with anomaly-related signals
<br> --- --- --- <br>
O notebook constrói uma comparação estruturada entre:
- um modelo baseline LightGBM treinado apenas com features engenheiradas
- um modelo híbrido LightGBM treinado com sinais relacionados a anomalias

### Inputs
- 📦`beverage_sales_feature.parquet`
- 📦`anomaly_predictions.parquet`
- 📊baseline trained model
- 📊hybrid trained model

### Main processing steps
1. Load features and anomaly datasets.
2. Convert `Order_Date` to datetime.
3. Create train and test subsets:
   - 2021–2022 for training comparison
   - 2023 for final comparison
4. Prepare feature lists for both approaches.
5. Use `ModelsMetricsReporter` to compare:
   - baseline model
   - hybrid model
6. Generate grouped performance tables, including comparison by anomaly flag.
7. Plot a vertical visual comparison of the models.
<br>
<br> --- --- --- <br>
1. Carregar os datasets de features e anomalias.
2. Converter `Order_Date` para datetime.
3. Criar subconjuntos de treino e teste:
   - 2021–2022 para comparação de treino
   - 2023 para comparação final
4. Preparar as listas de features para as duas abordagens.
5. Usar `ModelsMetricsReporter` para comparar:
   - modelo baseline
   - modelo híbrido
6. Gerar tabelas de desempenho agrupadas, incluindo comparação por flag de anomalia.
7. Plotar uma visualização vertical de comparação entre os modelos.

### Outputs
- 📝Benchmark report table
- 📊Comparison dataframe
- 📈Visualization of model performance
- ✅Evidence of whether the hybrid model performs better on anomalous periods or groups

### Why this notebook matters
This notebook gives business and technical justification for the hybrid architecture.  
It shows whether anomaly signals add measurable value instead of being included only by intuition.
<br> --- --- --- <br>
Este notebook fornece justificativa técnica e de negócio para a arquitetura híbrida.  
Ele mostra se os sinais de anomalia agregam valor mensurável, em vez de serem incluídos apenas por intuição.

### Notes
- The notebook compares models on both train-like and test datasets. For portfolio presentation, highlight the final test comparison as the most reliable result.
- The presence of grouped metrics by anomaly flag is useful because it shows whether the hybrid model is especially helpful in difficult or unusual demand scenarios.
<br> --- --- --- <br>
- O notebook compara os modelos tanto em dados de treino quanto em dados de teste. Para apresentação em portfólio, vale destacar a comparação final no conjunto de teste como o resultado mais confiável.
- A presença de métricas agrupadas por `anomaly_flag` é útil porque mostra se o modelo híbrido ajuda principalmente nos cenários mais difíceis ou incomuns de demanda.


In [12]:
%load_ext autoreload
%autoreload 2

import os
import glob
from pathlib import Path
import sys
import pyarrow
import pandas as pd


PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from src.config.config import DATA_PROCESSED, DATA_FEATURES, MODELS_BASELINE,MODELS_METRICS
from src.repository.parquet_repository import ParquetRepository
from src.models.LightGBMRegressorBaseLine import LightGBMRegressorBaseLine
from src.models.LightGBMRegressorAnomaly import LightGBMRegressorAnomaly
from src.visualization.ModelsMetricsReporter import ModelsMetricsReporter


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
repo = ParquetRepository(DATA_FEATURES)

In [3]:
df_features = repo.load("beverage_sales_feature.parquet")
df_anomalies = repo.load("anomaly_predictions.parquet")

df_features["Order_Date"] = pd.to_datetime(df_features["Order_Date"])
df_anomalies["Order_Date"] = pd.to_datetime(df_anomalies["Order_Date"])

[OK] Arquivo carregado: D:\PROJETOS\git_repo\BEVERAGE-SALES\data\features\beverage_sales_feature.parquet
[OK] Arquivo carregado: D:\PROJETOS\git_repo\BEVERAGE-SALES\data\features\anomaly_predictions.parquet


In [4]:
print (df_anomalies.dtypes)

Order_Date                  datetime64[ns]
Category                            object
Product                             object
Region                              object
quantity_sum                       float64
total_price_sum                    float64
unit_price_mean                    float64
discount_mean                      float64
order_count                        float64
customer_count                     float64
avg_ticket                         float64
day_of_week                          int32
month                                int32
year                                 int32
is_weekend                           int64
quantity_sum_mean_7d               float64
quantity_sum_std_7d                float64
quantity_sum_sum_7d                float64
total_price_sum_mean_7d            float64
total_price_sum_std_7d             float64
total_price_sum_mean_30d           float64
unit_price_mean_mean_7d            float64
discount_mean_mean_14d             float64
quantity_vs

In [5]:
df_model_with_anomaly = df_features.merge(
    df_anomalies,
    on=["Order_Date", "Product", "Region"],
    how="left"
)

In [6]:
print (df_anomalies.dtypes)

Order_Date                  datetime64[ns]
Category                            object
Product                             object
Region                              object
quantity_sum                       float64
total_price_sum                    float64
unit_price_mean                    float64
discount_mean                      float64
order_count                        float64
customer_count                     float64
avg_ticket                         float64
day_of_week                          int32
month                                int32
year                                 int32
is_weekend                           int64
quantity_sum_mean_7d               float64
quantity_sum_std_7d                float64
quantity_sum_sum_7d                float64
total_price_sum_mean_7d            float64
total_price_sum_std_7d             float64
total_price_sum_mean_30d           float64
unit_price_mean_mean_7d            float64
discount_mean_mean_14d             float64
quantity_vs

In [17]:
df_features["Order_Date"] = pd.to_datetime(df_features["Order_Date"])

df_train_features = df_features[df_features["Order_Date"].dt.year.isin([2021, 2022])].copy()
df_test_features = df_features[df_features["Order_Date"].dt.year == 2023].copy()

In [7]:
df_anomalies["Order_Date"] = pd.to_datetime(df_anomalies["Order_Date"])

df_train_anomaly = df_anomalies[df_anomalies["Order_Date"].dt.year.isin([2021, 2022])].copy()
df_test_anomaly = df_anomalies[df_anomalies["Order_Date"].dt.year == 2023].copy()

In [9]:
target_col = "quantity_sum"

numeric_features = [
    "total_price_sum",
    "unit_price_mean",
    "discount_mean",
    "order_count",
    "customer_count",
    "avg_ticket",
    "day_of_week",
    "month",
    "is_weekend",
    "quantity_sum_mean_7d",
    "quantity_sum_std_7d",
    "quantity_sum_sum_7d",
    "total_price_sum_mean_7d",
    "total_price_sum_std_7d",
    "total_price_sum_mean_30d",
    "unit_price_mean_mean_7d",
    "discount_mean_mean_14d",
    "quantity_vs_mean_7d",
    "total_price_vs_mean_30d",
    "quantity_pct_vs_mean_7d",
    "history_less_than_7d",
    "history_less_than_30d"
]

categorical_features = [
    "Product",
    "Region"
]

baseline_model = LightGBMRegressorBaseLine(
    target_col=target_col,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    model_dir=MODELS_BASELINE,
    random_state=3,
    n_splits=3,
    scoring="neg_root_mean_squared_error"
)

LOCAL_BASELINE = r"D:\PROJETOS\git_repo\BEVERAGE-SALES\models\baseline\lightgbm_baseline_model.joblib"
baseline_model.load_model(LOCAL_BASELINE)

In [10]:
target_col = "quantity_sum"

numeric_features = [
    "total_price_sum",
    "unit_price_mean",
    "discount_mean",
    "order_count",
    "customer_count",
    "avg_ticket",
    "day_of_week",
    "month",
    "is_weekend",
    "quantity_sum_mean_7d",
    "quantity_sum_std_7d",
    "quantity_sum_sum_7d",
    "total_price_sum_mean_7d",
    "total_price_sum_std_7d",
    "total_price_sum_mean_30d",
    "unit_price_mean_mean_7d",
    "discount_mean_mean_14d",
    "quantity_vs_mean_7d",
    "total_price_vs_mean_30d",
    "quantity_pct_vs_mean_7d",
    "history_less_than_7d",
    "history_less_than_30d",
    "anomaly_score",
    "anomaly_flag"
]

categorical_features = [
    "Product",
    "Region"
]

hybrid_model = LightGBMRegressorAnomaly(
    target_col=target_col,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    model_dir=MODELS_BASELINE,
    random_state=3,
    n_splits=3,
    scoring="neg_root_mean_squared_error"
)

LOCAL_HYBRID_MODEL = r"D:\PROJETOS\git_repo\BEVERAGE-SALES\models\baseline\lightgbm_anomalies_model.joblib"
hybrid_model.load_model(LOCAL_HYBRID_MODEL)

In [30]:
modelreport = ModelsMetricsReporter (df_baseline=df_train_features, df_with_anomaly=df_train_anomaly)

report, df_compare = modelreport.compare_models_by_anomaly_flag(
    baseline_model=baseline_model,
    anomaly_model=hybrid_model,
    target_col="quantity_sum",
    anomaly_flag_col="anomaly_flag",
    id_columns=["Order_Date", "Product", "Region"]
)

d:\PROJETOS\git_repo\BEVERAGE-SALES\_venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
d:\PROJETOS\git_repo\BEVERAGE-SALES\_venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [31]:
print(report)

   anomaly_flag   count  baseline_mae  anomaly_model_mae  baseline_median_ae  \
0             0  543389      2.692384           2.677233            1.916530   
1             1    5571      6.156137           6.387197            2.514491   

   anomaly_model_median_ae  baseline_rmse  anomaly_model_rmse  \
0                 1.895941       3.976524            3.971992   
1                 2.428869      12.726619           13.384858   

   baseline_max_ae  anomaly_model_max_ae  mae_improvement  rmse_improvement  \
0       158.320738            138.362552         0.015152          0.004532   
1       123.955175            114.566744        -0.231061         -0.658239   

   mae_improvement_pct  rmse_improvement_pct  
0             0.562767              0.113976  
1            -3.753338             -5.172145  


In [32]:
modelreport = ModelsMetricsReporter (df_baseline=df_test_features, df_with_anomaly=df_test_anomaly)

report, df_compare = modelreport.compare_models_by_anomaly_flag(
    baseline_model=baseline_model,
    anomaly_model=hybrid_model,
    target_col="quantity_sum",
    anomaly_flag_col="anomaly_flag",
    id_columns=["Order_Date", "Product", "Region"]
)

print(report)

d:\PROJETOS\git_repo\BEVERAGE-SALES\_venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
d:\PROJETOS\git_repo\BEVERAGE-SALES\_venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


   anomaly_flag   count  baseline_mae  anomaly_model_mae  baseline_median_ae  \
0             0  271071      2.777472           2.660743            1.938180   
1             1    2656      3.174978           3.078371            2.096747   

   anomaly_model_median_ae  baseline_rmse  anomaly_model_rmse  \
0                 1.891841       4.317620            3.949275   
1                 1.939884       5.717656            5.322033   

   baseline_max_ae  anomaly_model_max_ae  mae_improvement  rmse_improvement  \
0       263.094056            178.650378         0.116730          0.368345   
1        80.584547             67.309442         0.096607          0.395622   

   mae_improvement_pct  rmse_improvement_pct  
0             4.202728              8.531211  
1             3.042777              6.919310  
